# 练习一：读 CSV + groupby 聚合

> 任务三「AI 工具初体验」的 pandas 部分（1 / 2）
> 数据：`data/sales.csv` —— 60 条销售明细，4 个区域、5 种商品，2026 年 7-9 月
> 怎么跑：用 VS Code 打开本文件 → 右上角内核选 `dev-basics/.venv` → 逐格 `Shift + Enter`

## 这一节要回答三个问题

1. 这三个月一共卖了多少？
2. 哪个区域卖得最好，差多少？
3. 哪个月最差？

三个问题都指向同一个动作：**把 60 行明细压缩成几行汇总**。这就是 `groupby`。

## 先看清楚：groupby 到底在干什么

一句话：**拆分（split）→ 应用（apply）→ 合并（combine）**。

```
        60 行明细
            │
   ┌────────┼────────┐        ← 第一步：按 region 拆成 4 堆
  华东      华北      华南      西南
  (15 行)   (13 行)   (16 行)   (16 行)
   │         │         │         │
 求和       求和       求和       求和     ← 第二步：每一堆各算各的
   │         │         │         │
 42792     78949     34312     67561     ← 第三步：拼回一张小表
```

**你只需要说清楚两件事：按什么拆（`by=`）、拆完算什么（`agg=`）。**

In [1]:
from pathlib import Path

import pandas as pd

# 兼容两种打开方式：在项目根目录打开（路径是 pandas/data），
# 或者直接打开 pandas/ 目录（路径是 data）。两种都不用改代码。
DATA = Path("pandas/data")
if not DATA.exists():
    DATA = Path("data")

print("pandas", pd.__version__)
print("数据目录:", DATA.resolve())

pandas 3.0.6
数据目录: C:\Users\Administrator\WorkBuddy\2026-10-13-16-35-32\dev-basics\pandas\data


## 第 1 步：读进来

`pd.read_csv()` 只有一个必填参数：文件路径。

但有个参数最好一开始就加上：`encoding="utf-8-sig"`。

In [2]:
sales = pd.read_csv(DATA / "sales.csv", encoding="utf-8-sig")
print(sales.shape)          # (行数, 列数)
sales.head()                # 只看前 5 行，别急着 print 全表

(60, 5)


,order_date,region,product,quantity,unit_price
0,2026-09-10,华北,键盘,1,399.0
1,2026-07-08,华东,显示器,5,1299.0
2,2026-07-07,华北,显示器,3,1299.0
3,2026-07-24,华北,耳机,2,799.0
4,2026-07-07,西南,耳机,1,799.0


### 为什么建议加 `utf-8-sig`（一个实测出来的细节）

这个文件是**带 BOM 的 UTF-8**（`utf-8-sig` 写出来的）。BOM 是文件开头 3 个看不见的字节 `EF BB BF`，
作用是告诉 Excel「我是 UTF-8，别用 GBK 打开我」，否则中文在 Excel 里会乱码。

问题在于：有些工具读到它不会自动去掉，于是**第一个列名前面会多一个看不见的字符**，
你写 `sales["order_date"]` 就会报 `KeyError`——明明看着一模一样。

下面用标准库 `csv` 读同一个文件，看看它长什么样：

In [3]:
import csv

with open(DATA / "sales.csv", encoding="utf-8", newline="") as f:
    header = next(csv.reader(f))

print("第一列名:", repr(header[0]))
print("带 BOM 吗:", header[0].startswith("\ufeff"))

第一列名: '\ufefforder_date'
带 BOM 吗: True


看到了：`'\ufefforder_date'` —— 前面那个 `\ufeff` 就是 BOM。

**但有意思的是：pandas 3.0 会自动吃掉它。** 上面 `read_csv` 用 `utf-8` 读也不会出问题，
这是新版引擎做的兼容。所以现在：

- 用 **pandas** 读：加不加 `utf-8-sig` 都行（加了更保险，也能兼容旧版本）；
- 用 **标准库 csv / 其他工具**读：必须自己处理，否则第一列名是脏的。

这就是「实测比直觉可靠」的一个例子——我原本以为 pandas 一定会带 BOM，跑了一遍才发现不会。

## 第 2 步：认识数据（3 个命令够用）

在读数据之前，先问自己：**每列是什么类型？有没有异常值？**

- `.dtypes` —— 每列的类型
- `.describe()` —— 数值列的分布（最小、最大、均值、四分位）
- `.info()` —— 类型 + 有没有空值，一次看全

In [4]:
sales.dtypes

order_date        str
region            str
product           str
quantity        int64
unit_price    float64
dtype: object

In [5]:
sales.describe()

,quantity,unit_price
count,60.000000,60.000000
mean,2.766667,1337.833333
std,1.566240,1606.488002
min,1.000000,129.000000
25%,1.000000,399.000000
50%,2.000000,799.000000
75%,4.000000,1299.000000
max,5.000000,4999.000000


注意两点：

1. `order_date` 的类型是 **`str`（字符串）**，不是日期。因为 CSV 里存的就是文本，pandas 不会替你猜。
   （顺带一提：pandas 3.0 里字符串列的 dtype 显示为 `str`，老版本是 `object`，看到 `str` 别奇怪。）
2. `quantity` 是 `int64`，`unit_price` 是 `float64`——**这两个才是能算的**。

日期不转成 `datetime`，就没法按"月"分组。转换只要一行：

In [6]:
sales["order_date"] = pd.to_datetime(sales["order_date"])

print(sales["order_date"].dtype)
print("时间范围:", sales["order_date"].min().date(), "→", sales["order_date"].max().date())

datetime64[us]
时间范围: 2026-07-07 → 2026-09-28


## 第 3 步：先加一列"金额"

原始数据里**没有金额**，只有单价和数量。这是现实中的常态：明细表只存原子字段，派生字段自己算。

**关键习惯：先算成列，再 groupby。** 不要在 `agg` 里写 `quantity * unit_price`——能写，
但可读性差，而且一旦后面要复用这个金额，就得再算一遍。

In [7]:
sales["amount"] = sales["quantity"] * sales["unit_price"]

print("总金额:", sales["amount"].sum())
print("订单数:", len(sales))
print("平均每单:", round(sales["amount"].mean(), 2))

总金额: 223614.0
订单数: 60
平均每单: 3726.9


**这三个月一共卖了 223,614 元**——第一个问题答完了。

## 第 4 步：groupby 的三种写法

同一个「按区域求金额和」，有三种写法。**先都认识，再挑一种固定用。**

In [8]:
# 写法一：最直白。选列 → groupby → 聚合
by_region = sales.groupby("region")["amount"].sum()
by_region

region
华东    42792.0
华北    78949.0
华南    34312.0
西南    67561.0
Name: amount, dtype: float64

In [9]:
# 写法二：先 groupby 再选列（结果一样，只是顺序不同）
sales.groupby("region")["amount"].sum().equals(by_region)

True

In [10]:
# 写法三：命名聚合。一次算多个指标，而且能指定结果列名
sales.groupby("region").agg(
    total=("amount", "sum"),
    orders=("amount", "size"),
    avg=("amount", "mean"),
)

,total,orders,avg
region,,,
华东,42792.0,19,2252.210526
华北,78949.0,14,5639.214286
华南,34312.0,14,2450.857143
西南,67561.0,13,5197.000000


**建议从写法三开始用。** 理由：

- 结果直接是 DataFrame，列名你自己定（`total` / `orders` / `avg`），不用后面再改名；
- 一次能算多个指标，不用写三行再 `concat`；
- 写 `("amount", "sum")` 这种元组时，意图比 `["amount"].sum()` 更清楚。

第一个问题的延伸：**哪个区域最好？**

In [11]:
by_region.sort_values(ascending=False)

region
华北    78949.0
西南    67561.0
华东    42792.0
华南    34312.0
Name: amount, dtype: float64

华北 78,949，西南 67,561，华东 42,792，华南 34,312。
**最好的和最差的差 2.3 倍**——这个差距比"平均数"有信息量得多。

## 第 5 步：按多列分组

`groupby` 可以接一个列表，按组合拆。比如"每个区域的每种商品卖了多少"。

In [12]:
by_rp = sales.groupby(["region", "product"]).agg(
    total=("amount", "sum"),
    orders=("amount", "size"),
)

print("组合数:", len(by_rp))
by_rp.head(10)

组合数: 18


total  orders
region product                 
华东     显示器      27279.0       5
       耳机        9588.0       4
       键盘        3990.0       5
       鼠标        1935.0       5
华北     显示器      18186.0       4
       笔记本      49990.0       3
       耳机        9588.0       3
       键盘         798.0       2
       鼠标         387.0       2
华南     显示器       6495.0       3

4 个区域 × 5 种商品 = 理论上 20 种组合，实际只有 **18 种**——
说明有两种组合这三个月**一单都没卖**。这种"缺失的组合"在后面做透视表时很重要。

## 第 6 步：`reset_index` 的坑（新手最容易卡在这）

`groupby` 之后，**分组字段变成了索引（index），不再是普通列**。看：

In [13]:
s = sales.groupby("region")["amount"].sum()

print("类型:", type(s).__name__)
print("索引名:", s.index.name)
print("能不能直接取 region 列:", "region" in s.columns if hasattr(s, "columns") else "这是 Series，没有 columns")

类型: Series
索引名: region
能不能直接取 region 列: 这是 Series，没有 columns


In [14]:
# 想把它变回普通表格 → reset_index()
df = s.reset_index()
print(list(df.columns))
df

['region', 'amount']


,region,amount
0,华东,42792.0
1,华北,78949.0
2,华南,34312.0
3,西南,67561.0


**记住这条**：只要后面还要继续用分组字段（比如再 merge、再筛选），就先 `.reset_index()`。
不 reset 直接 `df["region"]` 会报 `KeyError`——因为它在 index 里，不在列里。

## 第 7 步：按月看趋势

有了 `datetime` 类型，就能用 `.dt` 取年月。

In [15]:
sales["month"] = sales["order_date"].dt.to_period("M")

by_month = sales.groupby("month")["amount"].sum()
by_month

month
2026-07    113313.0
2026-08     57282.0
2026-09     53019.0
Freq: M, Name: amount, dtype: float64

**7 月 113,313 → 8 月 57,282 → 9 月 53,019**，逐月下滑，9 月只有 7 月的一半不到。
第三个问题答完了。

> ⚠️ 这是**造出来的数据**，别当真业务结论。但它演示了一件事：
> **光看总数 223,614 什么都发现不了，拆成月份才看得出趋势。** 这就是 groupby 的价值。

## 第 8 步：透视表（一张表看两个维度）

`pivot_table` 可以理解成"把 groupby 的结果摆成二维表格"：
行是一个维度，列是另一个维度。

In [16]:
sales.pivot_table(
    index="region",
    columns="product",
    values="amount",
    aggfunc="sum",
    fill_value=0,
).round(0)

product,显示器,笔记本,耳机,键盘,鼠标
region,,,,,
华东,27279.0,0.0,9588.0,3990.0,1935.0
华北,18186.0,49990.0,9588.0,798.0,387.0
华南,6495.0,19996.0,4794.0,1995.0,1032.0
西南,12990.0,44991.0,3196.0,6384.0,0.0


两个细节：

- `fill_value=0`：刚才说的"18 种组合而不是 20 种"，空着的位置在这里填 0；
- `.round(0)`：金额显示成整数，好看。**注意它只改变显示，不改变存储的值。**

## 小结：这一节真正要记住的

**四个动作**：

| 动作 | 写法 |
|---|---|
| 读文件 | `pd.read_csv(path, encoding="utf-8-sig")` |
| 加派生列 | `df["新列"] = 表达式` |
| 分组聚合 | `df.groupby("列").agg(名字=("列", "函数"))` |
| 变回普通表 | `.reset_index()` |

**三个坑**：

| 坑 | 症状 | 解法 |
|---|---|---|
| 日期是字符串 | 没法 `.dt.month`，报 `AttributeError` | `pd.to_datetime()` |
| 分组字段变索引 | `KeyError: 'region'` | `.reset_index()` |
| BOM 藏在列名里 | `KeyError`，但列名看着一样 | 读的时候加 `encoding="utf-8-sig"` |

## 自己动手（3 个小任务）

1. 算**每个区域的订单数**，并按订单数从多到少排。
2. 算**每种商品的平均单价**（提示：用 `"mean"`，但注意要先按商品分组再对 `unit_price` 求均值——别对 `amount` 求）。
3. 找出**金额最大的那一单**（提示：`sales.loc[sales["amount"].idxmax()]`）。

> 验收标准不是"跑出结果"，是**能说出每个结果是怎么来的**。